# Reconocimiento de figuras con la cámara

Este notebook reutiliza las técnicas de los dos notebooks anteriores para reconocer una figura (triángulo, cuadrado, círculo, …) y su color a partir de la cámara:

| Paso | Técnica | Viene de |
| ---- | ------- | -------- |
| 1. Encontrar dónde está la figura | **Resta** de imágenes (`cv2.absdiff`) entre el fondo vacío y la escena con la figura | `3_OperacionesMatemáticas` |
| 2. Recortar solo la figura | Operación lógica **AND** (`cv2.bitwise_and`) con la máscara | `3_OperacionesMatemáticas` |
| 3. Saber de qué color es | **K-Means** sobre los píxeles de la figura: el centroide del grupo más grande es el color dominante | `4_Kmeans_Imagenes` |
| 4. Saber qué forma tiene | Contornos: se cuenta el número de vértices y se mide qué tan circular es | nuevo |

## ¿Cómo usarlo?

1. Pon la cámara **fija** (apoyada, sin moverla entre fotos) apuntando a un fondo liso.
2. Toma una foto del **fondo vacío**.
3. Coloca la figura (por ejemplo, recortada en papel de color) y toma otra foto.
4. El notebook resta las dos fotos: lo único que cambió es la figura.

Funciona en **Google Colab** (usa la cámara del navegador) y en **Jupyter local** (usa `cv2.VideoCapture`; ahí además hay una celda en tiempo real al final).

In [ ]:
import sys
import unicodedata

import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans


EN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Colab:", EN_COLAB)

## Configuración

* `INDICE_CAMARA`: qué cámara usar. En este equipo hay dos: **0** y **1**. La frontal es la **1**; si no es así, cámbialo a 0 (la celda de la sección 1 muestra una imagen de cada una para comprobarlo).
* `INTERVALO`: cada cuántos segundos se toma una foto en la cámara en vivo (sección 8).

In [ ]:
INDICE_CAMARA = 1
INTERVALO = 0.5  # segundos


def abrir_camara(indice=None):
    if indice is None:
        indice = INDICE_CAMARA

    # En Windows, CAP_DSHOW abre la cámara más rápido
    if sys.platform == "win32":
        camara = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
    else:
        camara = cv2.VideoCapture(indice)

    if not camara.isOpened():
        raise RuntimeError(f"No fue posible abrir la cámara {indice}.")

    return camara

# 1. Captura desde la cámara

- **En Colab** el código Python corre en un servidor de Google y no tiene acceso a tu cámara. Por eso se usa un pequeño programa en JavaScript que abre la cámara en el navegador, muestra un botón y devuelve la foto a Python.
- **En local** se usa `cv2.VideoCapture(0)`: se abre una ventana con la cámara y se captura con la tecla **ESPACIO**.

En ambos casos se muestra la vista previa antes de capturar, así la cámara tiene tiempo de ajustar la exposición.

In [ ]:
def tomar_foto_colab(mensaje="Capturar"):
    from base64 import b64decode

    from IPython.display import Javascript, display
    from google.colab.output import eval_js

    js = Javascript('''
    async function tomarFoto(mensaje) {
      const div = document.createElement('div');
      const boton = document.createElement('button');
      boton.textContent = mensaje;
      div.appendChild(boton);

      const video = document.createElement('video');
      video.style.display = 'block';
      // facingMode 'user' = cámara frontal
      const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'user'}});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Esperar a que se presione el botón
      await new Promise((resolve) => boton.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', 0.95);
    }
    ''')

    display(js)
    datos = eval_js(f'tomarFoto("{mensaje}")')

    # "data:image/jpeg;base64,...." -> bytes -> imagen BGR
    binario = b64decode(datos.split(",")[1])
    arreglo = np.frombuffer(binario, dtype=np.uint8)

    return cv2.imdecode(arreglo, cv2.IMREAD_COLOR)


def tomar_foto_local(mensaje="Capturar"):
    camara = abrir_camara()

    print(mensaje, "-> presiona ESPACIO para capturar (ESC para cancelar)")

    foto = None

    while True:
        ok, cuadro = camara.read()

        if not ok:
            break

        vista = cuadro.copy()
        cv2.putText(vista, mensaje, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("Camara", vista)

        tecla = cv2.waitKey(1) & 0xFF

        if tecla == 32:  # ESPACIO
            foto = cuadro
            break

        if tecla == 27:  # ESC
            break

    camara.release()
    cv2.destroyAllWindows()

    if foto is None:
        raise RuntimeError("No se capturó ninguna foto.")

    return foto


def tomar_foto(mensaje="Capturar"):
    if EN_COLAB:
        return tomar_foto_colab(mensaje)

    return tomar_foto_local(mensaje)

Para comprobar cuál es la cámara frontal (solo local), esta celda toma una imagen de cada cámara:

In [ ]:
if EN_COLAB:
    print("En Colab se usa la cámara frontal del navegador (facingMode 'user').")
else:
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

    for indice, eje in zip([0, 1], ejes):
        try:
            camara = abrir_camara(indice)

            # Los primeros cuadros suelen salir oscuros mientras se ajusta la exposición
            for _ in range(10):
                ok, cuadro = camara.read()

            camara.release()

            if not ok:
                raise RuntimeError("La cámara no devolvió imagen.")

            eje.imshow(cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB))
        except RuntimeError:
            eje.text(0.5, 0.5, "No disponible", ha="center")

        titulo = f"Cámara {indice}"
        if indice == INDICE_CAMARA:
            titulo += "  <- seleccionada"

        eje.set_title(titulo)
        eje.axis("off")

    plt.show()

# 2. Resta: encontrar la figura

Igual que en el ejemplo de "Diferencias Detectadas" del notebook 3:

```python
diferencia = cv2.absdiff(fondo, escena)
```

Las zonas que no cambiaron dan ≈ 0 y la figura queda resaltada. Con fotos reales hay que agregar tres cosas:

* **Suavizado** (`GaussianBlur`) antes de restar, porque la cámara siempre tiene algo de ruido.
* **Máximo de los canales** en lugar de pasar a gris: al pasar a gris el canal azul pesa solo 0.114, y un cambio que es solo de azul (por ejemplo, blanco → amarillo) da ≈ 29 y se pierde con el umbral.
* **Morfología** (apertura y cierre) para borrar puntos sueltos y rellenar huecos pequeños.

Al final se conserva solo el contorno más grande: esa es la figura.

In [ ]:
def mascara_por_resta(fondo, escena, umbral=40):
    fondo_suave = cv2.GaussianBlur(fondo, (7, 7), 0)
    escena_suave = cv2.GaussianBlur(escena, (7, 7), 0)

    # Resta (notebook 3)
    diferencia = cv2.absdiff(fondo_suave, escena_suave)

    # Cambio máximo en cualquiera de los canales B, G, R
    cambio = diferencia.max(axis=2)

    _, mascara = cv2.threshold(cambio, umbral, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel, iterations=2)

    return diferencia, mascara


def contorno_principal(mascara, area_minima=1500):
    contornos, _ = cv2.findContours(
        mascara,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contornos = [c for c in contornos if cv2.contourArea(c) >= area_minima]

    if not contornos:
        return None

    return max(contornos, key=cv2.contourArea)

# 3. AND: recortar la figura

Con el contorno de la figura se dibuja una máscara rellena (blanco = figura, negro = resto) y se aplica AND, igual que con la máscara de círculos del paisaje en el notebook 3:

* blanco (`255`) → se conserva el píxel
* negro (`0`) → el píxel desaparece

In [ ]:
def recortar_figura(escena, contorno):
    mascara_figura = np.zeros(escena.shape[:2], dtype=np.uint8)
    cv2.drawContours(mascara_figura, [contorno], -1, 255, -1)

    # Máscara de 3 canales, como la imagen de círculos del notebook 3
    mascara_3c = cv2.merge([mascara_figura] * 3)

    figura = cv2.bitwise_and(escena, mascara_3c)

    return mascara_figura, figura

# 4. K-Means: color de la figura

Es el mismo procedimiento del notebook 4, pero solo con los píxeles de la figura:

1. Se toman los píxeles donde la máscara es blanca → tabla de `N × 3`.
2. K-Means agrupa esos colores en `K` grupos. Aunque la figura sea de un solo color, la iluminación crea brillos y sombras; por eso se usa K = 3.
3. El centroide del grupo **con más píxeles** es el color dominante.

Para que sea rápido con fotos grandes se entrena con una muestra de píxeles y después se predice el grupo de todos.

Para darle nombre al color se pasa a HSV, donde el **tono** (H) indica el color sin importar si hay más o menos luz.

In [ ]:
def color_dominante(escena, mascara_figura, K=3, n_muestras=5000, n_init=10):
    escena_rgb = cv2.cvtColor(escena, cv2.COLOR_BGR2RGB)

    # Tabla de píxeles de la figura (notebook 4)
    pixeles = escena_rgb[mascara_figura > 0].astype(np.float32)

    K = min(K, len(pixeles))

    generador = np.random.default_rng(42)
    tamano = min(n_muestras, len(pixeles))
    muestra = pixeles[generador.choice(len(pixeles), tamano, replace=False)]

    modelo = KMeans(n_clusters=K, random_state=42, n_init=n_init)
    modelo.fit(muestra)

    etiquetas = modelo.predict(pixeles)
    colores = np.round(modelo.cluster_centers_).astype(np.uint8)

    # Grupo con más píxeles = color dominante
    conteo = np.bincount(etiquetas, minlength=K)
    color = colores[conteo.argmax()]

    # Figura segmentada: cada píxel reemplazado por su centroide (notebook 4)
    segmentada = np.zeros_like(escena_rgb)
    segmentada[mascara_figura > 0] = colores[etiquetas]

    return color, segmentada


def nombre_color(color_rgb):
    h, s, v = cv2.cvtColor(np.uint8([[color_rgb]]), cv2.COLOR_RGB2HSV)[0, 0]

    # En OpenCV H va de 0 a 179 (grados / 2)
    if v < 50:
        return "negro"

    if s < 50:
        return "blanco" if v > 190 else "gris"

    if h < 8 or h >= 165:
        return "rojo"
    if h < 20:
        return "naranja"
    if h < 35:
        return "amarillo"
    if h < 85:
        return "verde"
    if h < 100:
        return "cian"
    if h < 130:
        return "azul"
    if h < 150:
        return "morado"

    return "rosado"

# 5. Forma de la figura

Con el contorno se calculan dos medidas:

* **Número de vértices**: `cv2.approxPolyDP` simplifica el contorno a un polígono con pocos lados. Un triángulo queda con 3 vértices, un cuadrado con 4, etc.
* **Circularidad**:

$$\text{circularidad} = \frac{4\pi \cdot \text{área}}{\text{perímetro}^2}$$

  Vale 1 para un círculo perfecto y es menor para cualquier otra forma (cuadrado ≈ 0.785, triángulo equilátero ≈ 0.605).

Para distinguir cuadrado de rectángulo se usa `cv2.minAreaRect`, que mide el ancho y el alto aunque la figura esté girada.

In [ ]:
def clasificar_forma(contorno):
    perimetro = cv2.arcLength(contorno, True)
    area = cv2.contourArea(contorno)

    aproximado = cv2.approxPolyDP(contorno, 0.02 * perimetro, True)
    vertices = len(aproximado)

    circularidad = 4 * np.pi * area / perimetro ** 2

    if vertices == 3:
        forma = "triángulo"

    elif vertices == 4:
        (_, _), (ancho, alto), _ = cv2.minAreaRect(contorno)
        proporcion = min(ancho, alto) / max(ancho, alto)
        forma = "cuadrado" if proporcion > 0.85 else "rectángulo"

    elif vertices == 5:
        forma = "pentágono"

    elif vertices == 6:
        forma = "hexágono"

    elif circularidad > 0.85:
        forma = "círculo"

    elif circularidad > 0.6:
        forma = "óvalo"

    else:
        forma = "irregular"

    return forma, aproximado, circularidad

# 6. Juntar todo

In [ ]:
def sin_tildes(texto):
    # cv2.putText no dibuja tildes: "triángulo" -> "triangulo"
    texto = unicodedata.normalize("NFKD", texto)
    return texto.encode("ascii", "ignore").decode()


def reconocer_figura(fondo, escena, umbral=40, K=3, n_init=10):
    diferencia, mascara = mascara_por_resta(fondo, escena, umbral)
    contorno = contorno_principal(mascara)

    if contorno is None:
        return None

    mascara_figura, figura = recortar_figura(escena, contorno)
    color_rgb, segmentada = color_dominante(escena, mascara_figura, K=K, n_init=n_init)
    forma, aproximado, circularidad = clasificar_forma(contorno)
    color = nombre_color(color_rgb)

    # Imagen anotada
    anotada = escena.copy()
    cv2.drawContours(anotada, [aproximado], -1, (0, 0, 255), 3)

    x, y, _, _ = cv2.boundingRect(contorno)
    texto = sin_tildes(f"{forma} {color}")
    cv2.putText(anotada, texto, (x, max(y - 10, 25)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 4)
    cv2.putText(anotada, texto, (x, max(y - 10, 25)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    return {
        "forma": forma,
        "color": color,
        "color_rgb": color_rgb,
        "vertices": len(aproximado),
        "circularidad": circularidad,
        "aproximado": aproximado,
        "diferencia": diferencia,
        "mascara": mascara,
        "figura": figura,
        "segmentada": segmentada,
        "anotada": anotada,
    }


def mostrar_resultado(fondo, escena, resultado):
    paneles = [
        (cv2.cvtColor(fondo, cv2.COLOR_BGR2RGB), "Fondo"),
        (cv2.cvtColor(escena, cv2.COLOR_BGR2RGB), "Escena con figura"),
        (resultado["mascara"], "Resta + umbral"),
        (cv2.cvtColor(resultado["figura"], cv2.COLOR_BGR2RGB), "AND con la máscara"),
        (resultado["segmentada"], "K-Means sobre la figura"),
        (cv2.cvtColor(resultado["anotada"], cv2.COLOR_BGR2RGB), "Resultado"),
    ]

    fig, ejes = plt.subplots(2, 3, figsize=(18, 10))

    for eje, (imagen, titulo) in zip(ejes.ravel(), paneles):
        eje.imshow(imagen, cmap="gray" if imagen.ndim == 2 else None)
        eje.set_title(titulo)
        eje.axis("off")

    plt.tight_layout()
    plt.show()

    print("Forma:", resultado["forma"])
    print("Color:", resultado["color"], "RGB =", tuple(int(c) for c in resultado["color_rgb"]))
    print("Vértices:", resultado["vertices"])
    print("Circularidad:", round(resultado["circularidad"], 3))

# 7. Prueba sin cámara

Antes de usar la cámara se comprueba el procedimiento con imágenes dibujadas, como en el notebook 3. Al fondo se le agrega ruido para imitar una cámara real, y algunas figuras están giradas.

In [ ]:
def fondo_simulado(alto=480, ancho=640, semilla=0):
    generador = np.random.default_rng(semilla)
    fondo = np.full((alto, ancho, 3), 210, dtype=np.int16)
    fondo += generador.integers(-8, 9, size=fondo.shape, dtype=np.int16)
    return np.clip(fondo, 0, 255).astype(np.uint8)


def poligono_regular(centro, radio, lados, angulo=0):
    t = np.deg2rad(angulo) + 2 * np.pi * np.arange(lados) / lados
    puntos = np.stack([centro[0] + radio * np.cos(t), centro[1] + radio * np.sin(t)], axis=1)
    return puntos.astype(np.int32)


# (nombre esperado, función que dibuja la figura, color BGR)
pruebas = [
    ("triángulo", lambda img, c: cv2.fillPoly(img, [poligono_regular((320, 250), 120, 3, -90)], c), (0, 0, 220)),
    ("cuadrado", lambda img, c: cv2.fillPoly(img, [poligono_regular((320, 240), 120, 4, 20)], c), (200, 60, 0)),
    ("rectángulo", lambda img, c: cv2.rectangle(img, (180, 170), (460, 310), c, -1), (0, 170, 0)),
    ("pentágono", lambda img, c: cv2.fillPoly(img, [poligono_regular((320, 240), 120, 5, -90)], c), (0, 220, 255)),
    ("hexágono", lambda img, c: cv2.fillPoly(img, [poligono_regular((320, 240), 120, 6, 10)], c), (160, 0, 140)),
    ("círculo", lambda img, c: cv2.circle(img, (320, 240), 110, c, -1), (0, 130, 255)),
]

fondo = fondo_simulado(semilla=0)

fig, ejes = plt.subplots(2, 3, figsize=(18, 10))

for i, (esperado, dibujar, color) in enumerate(pruebas):
    escena = fondo_simulado(semilla=i + 1)  # otro ruido, como otra foto
    dibujar(escena, color)

    resultado = reconocer_figura(fondo, escena)

    acierto = "✔" if resultado["forma"] == esperado else "✘"

    eje = ejes.ravel()[i]
    eje.imshow(cv2.cvtColor(resultado["anotada"], cv2.COLOR_BGR2RGB))
    eje.set_title(f"{acierto} esperado: {esperado} | detectado: {resultado['forma']} {resultado['color']}")
    eje.axis("off")

plt.tight_layout()
plt.show()

Detalle de todos los pasos para una de las figuras de prueba:

In [ ]:
escena = fondo_simulado(semilla=99)
cv2.fillPoly(escena, [poligono_regular((320, 250), 120, 3, -90)], (0, 0, 220))

resultado = reconocer_figura(fondo, escena)
mostrar_resultado(fondo, escena, resultado)

# 8. Cámara en vivo: revisa una foto cada 0.5 segundos (local)

Al ejecutar la celda se abre la ventana **Camara** con el video en vivo:

1. Durante los primeros **2 segundos** deja la escena **vacía**: en ese momento se captura el fondo automáticamente.
2. Después, cada `INTERVALO` segundos (0.5 s) se toma una foto, se reconoce la figura y arriba de la ventana aparece qué es, por ejemplo **"Es: triangulo rojo"**, con la figura marcada en rojo.
3. Si la cámara se movió o cambió la luz, vuelve a dejar la escena vacía y presiona **F** para capturar el fondo de nuevo.
4. Para salir presiona **Q** o **ESC**.

Las teclas funcionan cuando la ventana **Camara** está seleccionada (haz clic sobre ella). A veces la ventana se abre detrás de VS Code: búscala en la barra de tareas.

En el notebook también se imprime cada resultado con su hora; cuando la figura cambia, la línea se marca con `<- cambió`.

Consejos:

* Pon la cámara quieta y usa figuras de papel de un color que contraste con el fondo.
* Si aparecen manchas de sombra o de luz como figura, sube `UMBRAL` (por ejemplo, 50 o 60).
* Si la figura queda incompleta, bájalo (por ejemplo, 25 o 30).

In [ ]:
import time
from datetime import datetime

ESPERA_FONDO = 2.0  # segundos con la escena vacía al empezar
UMBRAL = 40


def escribir_arriba(imagen, texto, color):
    # Franja negra arriba con el texto
    cv2.rectangle(imagen, (0, 0), (imagen.shape[1], 45), (0, 0, 0), -1)
    cv2.putText(imagen, sin_tildes(texto), (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)


if EN_COLAB:
    print("Esta celda solo funciona en Jupyter local. En Colab usa la sección 9.")
else:
    camara = abrir_camara()

    inicio = time.monotonic()
    fondo_vivo = None
    ultimo = None           # último resultado de reconocer_figura
    ultima_foto = 0.0       # momento de la última foto revisada
    anterior = None         # (forma, color) de la foto anterior

    print(f"Deja la escena vacía {ESPERA_FONDO:.0f} s para capturar el fondo...")

    while True:
        ok, cuadro = camara.read()

        if not ok:
            print("La cámara dejó de enviar imágenes.")
            break

        ahora = time.monotonic()

        # Fondo automático al empezar
        if fondo_vivo is None and ahora - inicio >= ESPERA_FONDO:
            fondo_vivo = cuadro.copy()
            print(f"Fondo capturado. Revisando una foto cada {INTERVALO} s (Q para salir)")

        # Cada INTERVALO segundos: tomar la foto y reconocerla
        if fondo_vivo is not None and ahora - ultima_foto >= INTERVALO:
            ultima_foto = ahora

            ultimo = reconocer_figura(fondo_vivo, cuadro.copy(), umbral=UMBRAL, n_init=1)

            hora = datetime.now().strftime("%H:%M:%S.%f")[:-4]

            if ultimo is None:
                actual = None
                linea = f"[{hora}] sin figura"
            else:
                actual = (ultimo["forma"], ultimo["color"])
                linea = f"[{hora}] {ultimo['forma']} {ultimo['color']}"

            if actual != anterior:
                linea += "  <- cambió"

            print(linea)
            anterior = actual

        # Ventana: video en vivo + último resultado arriba
        vista = cuadro.copy()

        if fondo_vivo is None:
            restante = ESPERA_FONDO - (ahora - inicio)
            escribir_arriba(vista, f"Escena vacia... fondo en {restante:.1f} s", (0, 255, 255))
        elif ultimo is None:
            escribir_arriba(vista, "Sin figura", (0, 0, 255))
        else:
            cv2.drawContours(vista, [ultimo["aproximado"]], -1, (0, 0, 255), 3)
            escribir_arriba(vista, f"Es: {ultimo['forma']} {ultimo['color']}", (0, 255, 0))

        cv2.imshow("Camara", vista)

        tecla = cv2.waitKey(1) & 0xFF

        if tecla == ord("f"):
            fondo_vivo = cuadro.copy()
            ultimo = None
            anterior = None
            print("Fondo capturado de nuevo.")

        if tecla in (ord("q"), 27):
            break

    camara.release()
    cv2.destroyAllWindows()

# 9. Foto a foto (Colab)

En Colab no hay video en vivo hacia Python, así que se hace con dos fotos tomadas con un botón (también funciona en local, con la tecla ESPACIO):

1. Ejecuta la celda y captura el **fondo vacío** (el navegador pedirá permiso para usar la cámara).
2. Coloca la figura **sin mover la cámara** y captura la escena.

In [ ]:
fondo_camara = tomar_foto("Capturar FONDO vacío")
print("Fondo capturado:", fondo_camara.shape)

In [ ]:
escena_camara = tomar_foto("Capturar ESCENA con la figura")

resultado = reconocer_figura(fondo_camara, escena_camara, umbral=40)

if resultado is None:
    print("No se encontró ninguna figura. Revisa que la cámara no se haya movido o baja el umbral.")
else:
    mostrar_resultado(fondo_camara, escena_camara, resultado)

Puedes volver a ejecutar la celda anterior con otra figura: el fondo se sigue usando mientras la cámara no se mueva.